In [11]:
from latency_evt_tools_clean import (
    audit_fixed,
    throughput_from_parquet_dir_fixed, summarize_throughput,
    simple_merge_metrics_fixed, collect_block_maxima_fixed, fit_gev, gev_quantiles
)

root = "../results/20250918_165513/consumer/consumer-sts-0_consumer-result/"
#root = "../results/20250921_210953/consumer/consumer-sts-0_consumer-result/"

# 1) Audit the whole scope (epoch seconds)
print(audit_fixed(root, time_col="consumer_receive_timestamp", size_col="size_bytes", time_unit="s"))

# 2) Throughput on an even sample across the run
ts = throughput_from_parquet_dir_fixed(
    root,
    time_col="consumer_receive_timestamp",
    size_col="size_bytes",
    resample="1S",
    time_unit="s",
    max_files=None,        # sample for speed
    sample="even"
)
print(summarize_throughput(ts))

# 3) Simple merge-like metrics + GEV
m = simple_merge_metrics_fixed(
    root,
    time_col="consumer_receive_timestamp",
    latency_col="end_to_end_latency_seconds",
    size_col="size_bytes",
    time_unit="s",
    max_files=2000,        # or None for all
    sample="even"
)
print(m["latency"], m["time_range"], m["throughput_mean"], m["gev"]["params"])


KeyboardInterrupt: 

In [12]:
from latency_evt_tools_clean import audit_fixed_parallel

#root = "../results/20250918_165513/consumer/consumer-sts-0_consumer-result/"
#root = "../results/20250921_210953/consumer/consumer-sts-0_consumer-result/"
audit = audit_fixed_parallel(
    root,
    time_col="consumer_receive_timestamp",
    size_col="size_bytes",
    time_unit="s",
    workers=12,           
    progress_every=2000,
)
print(audit)


[audit-parallel] 2000/142833 files
[audit-parallel] 4000/142833 files
[audit-parallel] 6000/142833 files
[audit-parallel] 8000/142833 files
[audit-parallel] 10000/142833 files
[audit-parallel] 12000/142833 files
[audit-parallel] 14000/142833 files
[audit-parallel] 16000/142833 files
[audit-parallel] 18000/142833 files
[audit-parallel] 20000/142833 files
[audit-parallel] 22000/142833 files
[audit-parallel] 24000/142833 files
[audit-parallel] 26000/142833 files
[audit-parallel] 28000/142833 files
[audit-parallel] 30000/142833 files
[audit-parallel] 32000/142833 files
[audit-parallel] 34000/142833 files
[audit-parallel] 36000/142833 files
[audit-parallel] 38000/142833 files
[audit-parallel] 40000/142833 files
[audit-parallel] 42000/142833 files
[audit-parallel] 44000/142833 files
[audit-parallel] 46000/142833 files
[audit-parallel] 48000/142833 files
[audit-parallel] 50000/142833 files
[audit-parallel] 52000/142833 files
[audit-parallel] 54000/142833 files
[audit-parallel] 56000/142833 fi

In [ ]:
from latency_evt_tools_clean import audit_fixed_multiprocess

audit = audit_fixed_multiprocess(
    root = root,
    time_col="consumer_receive_timestamp",
    size_col="size_bytes",
    time_unit="s",          # your timestamps are epoch seconds
    max_files=None,         # all files in scope
    sample="head",          # or "even"
    per_dir=None,           # or use N-per-subdir sampling
    group_level=1,
    processes=None,         # default: os.cpu_count()
    chunksize=256,          # tune 128–1024 depending on file size
    progress_every=2000,    # print progress every N files
)